In [ ]:
Sys.setenv(SP_EVOLUTION_STATUS = "2")
set.seed(1)

library(SeuratObject)
library(Matrix)
library(dplyr)
library(tidyr)
library(readr)
library(Seurat)
# library(DESeq2)

In [ ]:
# ---- 1. 参数设置 ------------------------------------------------------------
date <- "2026-07-28_adult"
result_dir <- "/project/work/2026-06-10_adult_DEG/Data/"

In [ ]:
# TODO: 请根据实际数据路径修改
seurat_rds_path <- '/project/work/2026-05-25_create_summary/Data/2026-06-05_new_adult_human_harmony_anno.rds'

min_cells <- 30      # 每个 pseudobulk 组的最小细胞数
max_cell <- Inf     # 每个 pseudobulk 组的最大细胞数（降采样上限）
padj_cutoff <- 0.05
logfc_cutoff <- 0.5

In [ ]:
# 全局统一的细胞类型顺序（与胚胎保持一致，去掉 RPC）
celltype_order <- c("Rod", "Cone", "AC", "BC", "HC", "MG", "RGC")

# 定义 stage
young_ages <- c("21years", "25years",'30years','32years')
old_ages <- c("67years", "75years", "86years")
unpaired_old_ages <- c("70years")  # 无 peripheral 对照，仅 Central-only 描述

In [ ]:
# ---- 2. 读取数据 ------------------------------------------------------------
cat("=== 1. 读取成年 snRNA Seurat 对象 ===\n")

obj <- readRDS(seurat_rds_path)
cat("  细胞数:", ncol(obj), "\n")
cat("  基因数:", nrow(obj), "\n")
cat("  metadata 字段:", paste(colnames(CONTACT_EMAIL), collapse = ", "), "\n")

In [ ]:
obj <- subset(obj,celltype %in% celltype_order)

In [ ]:
# 检查必要字段
required_cols <- c("celltype", "donor_age", "donor_id", "tissue")
missing_cols <- setdiff(required_cols, colnames(CONTACT_EMAIL))
if (length(missing_cols) > 0) {
  stop("metadata 缺少必要字段: ", paste(missing_cols, collapse = ", "))
}

In [ ]:
# 确保 celltype 因子顺序与项目统一
obj$celltype <- factor(obj$celltype, levels = celltype_order)

In [ ]:
# ---- 3. 添加 stage 和 tissue_group --------------------------------------
cat("\n=== 2. 定义 stage 和 tissue_group ===\n")

In [ ]:
CONTACT_EMAIL <- CONTACT_EMAIL %>%
  mutate(
    # tissue_group: fovea + macula -> Central, peripheral 保持不变
    tissue_group = case_when(
      tissue %in% c("fovea", "macula") ~ "Central",
      tissue == "peripheral" ~ "peripheral",
      TRUE ~ NA_character_
    ),
    tissue_group = factor(tissue_group, levels = c("peripheral", "Central")),
    
    # stage: Young / Old / Old_unpaired
    stage = case_when(
      donor_age %in% young_ages ~ "Young",
      donor_age %in% old_ages ~ "Old",
      donor_age %in% unpaired_old_ages ~ "Old_unpaired",
      TRUE ~ NA_character_
    )
  )

In [ ]:
# 检查 stage 定义情况
age_summary <- CONTACT_EMAIL %>%
  count(donor_age, stage, tissue_group, name = "n_cells") %>%
  arrange(stage, donor_age, tissue_group)
cat("  stage / tissue_group 分布:\n")
print(age_summary)

In [ ]:
# ---- 4. 配对完整性检查 ------------------------------------------------------
cat("\n=== 3. 配对完整性检查（celltype × stage × donor_age × tissue_group）===\n")

cell_counts <- CONTACT_EMAIL %>%
  count(celltype, stage, donor_age, donor_id, tissue_group, name = "n_cells") %>%
  arrange(celltype, stage, donor_age, tissue_group)

In [ ]:
# 保存原始细胞数报告
dir.create(result_dir, recursive = TRUE, showWarnings = FALSE)
write.csv(
  cell_counts,
  paste0(result_dir, date, "_cell_counts_by_group.csv"),
  row.names = FALSE
)
cat("  细胞数报告已保存: ", paste0(result_dir, date, "_cell_counts_by_group.csv"), "\n")

In [ ]:
# 生成配对完整性表
pairing_summary <- cell_counts %>%
  pivot_wider(
    id_cols = c(celltype, stage, donor_age, donor_id),
    names_from = tissue_group,
    values_from = n_cells,
    values_fill = 0
  ) %>%
  mutate(
    has_Central = Central > 0,
    has_peripheral = peripheral > 0,
    paired = has_Central & has_peripheral,
    min_n_cells = pmin(Central, peripheral)
  ) %>%
  select(celltype, stage, donor_age, donor_id, Central, peripheral, has_Central, has_peripheral, paired, min_n_cells) %>%
  arrange(celltype, stage, donor_age)

write.csv(
  pairing_summary,
  paste0(result_dir, date, "_pairing_completeness.csv"),
  row.names = FALSE
)
cat("  配对完整性表已保存: ", paste0(result_dir, date, "_pairing_completeness.csv"), "\n")

In [ ]:
cat("  可配对 donor 统计:\n")
print(pairing_summary %>% count(celltype, stage, paired, name = "n_donors"))

In [ ]:
# 标记低细胞数组合
low_count <- cell_counts %>% filter(n_cells < min_cells)
if (nrow(low_count) > 0) {
  cat("\n  警告: 以下组合细胞数 <", min_cells, ", 差异分析可能不可靠:\n")
  print(low_count)
  write.csv(
    low_count,
    paste0(result_dir, date, "_low_count_groups.csv"),
    row.names = FALSE
  )
} else {
  cat("\n  所有组合细胞数均 >=", min_cells, "\n")
}

In [ ]:
# ---- 5. 构建 Regional Pseudobulk (celltype × stage × donor_age × tissue_group) ------
cat("\n=== 4. 构建 Regional Pseudobulk ===\n")

celltypes <- levels(CONTACT_EMAIL$celltype)
celltypes <- celltypes[celltypes %in% unique(obj$celltype)]  # 只保留实际存在的 celltype

dir.create(paste0(result_dir, "/01_pseudobulk/regional"), recursive = TRUE, showWarnings = FALSE)

In [ ]:
for (ct in celltypes) {
  cat("  处理 celltype:", ct, "\n")
  
  sub <- subset(obj, subset = celltype == ct)
  
  if (ncol(sub) == 0) {
    cat("    跳过: 该 celltype 无细胞\n")
    next
  }
  
  sub$regional_group <- paste("pb", sub$donor_age, sub$tissue_group, sep = "-")
  
  # ---- 过滤低细胞数组合 + 全局上限降采样 ----
  meta_df <- CONTACT_EMAIL
  meta_df$cell_id <- rownames(meta_df)
  meta_df <- as_tibble(meta_df)
  
  # 1. 过滤 min_cells
  meta_df <- meta_df %>%
    group_by(regional_group) %>%
    filter(n() >= min_cells) %>%
    ungroup()
  
  # 2. 全局上限降采样
  if (nrow(meta_df) > 0) {
    set.seed(1)
    meta_df <- meta_df %>%
      group_by(regional_group) %>%
      sample_n(size = min(n(), max_cell)) %>%
      ungroup()
    sub <- sub[, meta_df$cell_id]
  }
  # ---- 过滤结束 ----
  
  cell_n <- CONTACT_EMAIL %>%
    count(regional_group, name = "n_cells")
  
  agg <- AggregateExpression(
    sub,
    group.by = "regional_group",
    assays = "RNA",
    return.seurat = FALSE,
    verbose = FALSE
  )
  pb_counts <- agg$RNA
  
  colData <- CONTACT_EMAIL %>%
    distinct(regional_group, stage, donor_age, donor_id, tissue_group) %>%
    left_join(cell_n, by = "regional_group") %>%
    mutate(
      stage = factor(stage),
      donor_age = factor(donor_age),
      donor_id = factor(donor_id),
      tissue_group = factor(tissue_group, levels = c("peripheral", "Central"))
    ) %>%
    select(group = regional_group, stage, donor_age, donor_id, tissue_group, n_cells)
  rownames(colData) <- colData$group
  
  colData <- colData[colnames(pb_counts), , drop = FALSE]
  
  if (!all(colnames(pb_counts) == rownames(colData))) {
    stop("colData 与 pb_counts 列名不匹配: ", ct)
  }
  
  saveRDS(
    list(counts = pb_counts, colData = colData),
    file = paste0(result_dir, "/01_pseudobulk/regional/", date, "_", ct, "_regional_pseudobulk.rds")
  )
  
  write.csv(
    colData,
    paste0(result_dir, "/01_pseudobulk/regional/", date, "_", ct, "_regional_colData.csv"),
    row.names = FALSE
  )
  
  cat("    完成: ", ncol(pb_counts), " 个 pseudobulk 样本\n")
}

In [ ]:
cat("\n========================================\n")
cat("Pseudobulk 构建完成！\n")
cat("========================================\n")